In [ ]:
# ===================================================================
# 🚀 V6 '헤드헌터(Head-Hunter)' 모델 (YOLO11n) - 점(Point) 기반 초정밀 탐지
# ===================================================================

import os
import glob
import json
import shutil
from tqdm.notebook import tqdm
from google.colab import drive

# 1. 구글 드라이브 연결
drive.mount('/content/drive')

# ===================================================================
# 🧹 [Step 0] 과거 데이터 초기화 및 순수 망원 환경 세팅
# ===================================================================
print("🧹 0. 기존 데이터 싹 다 지우고 백지에서 시작합니다...")
!rm -rf /content/extracted
!rm -rf /content/Jagalchi_YOLO

print("📦 1. 망원/중앙시장(CrowdData) 압축 해제 중...")
!unzip -q /content/drive/MyDrive/CrowdData.zip -d /content/extracted

raw_json_paths = glob.glob('/content/extracted/**/raw_jsons', recursive=True)
if not raw_json_paths:
    raise Exception("압축 파일 내부에 raw_jsons 폴더가 없습니다!")
BASE_DIR = os.path.dirname(raw_json_paths[0])
print(f"🚀 작업 폴더: {BASE_DIR}")

json_base_dir = os.path.join(BASE_DIR, 'raw_jsons')
image_base_dir = os.path.join(BASE_DIR, 'datasets', 'images')
label_base_dir = os.path.join(BASE_DIR, 'datasets', 'labels')

# ===================================================================
# 🎯 [Step 1] 핵심: 몸통 박스 버리고 '머리통 미니 박스(점)'로 변환!
# ===================================================================
def convert_to_head_hunter(json_dir, txt_dir, desc_name):
    os.makedirs(txt_dir, exist_ok=True)
    json_files = [f for f in os.listdir(json_dir) if f.endswith('.json')]

    for json_file in tqdm(json_files, desc=f"🎯 {desc_name} 머리통 박스(점) 생성 중"):
        try:
            json_filepath = os.path.join(json_dir, json_file)
            with open(json_filepath, 'r', encoding='utf-8-sig') as f:
                data = json.load(f)

            txt_filepath = os.path.join(txt_dir, json_file.replace('.json', '.txt'))
            img_width, img_height = 1920, 1080
            objects = data.get('image', {}).get('crowdinfo', {}).get('objects', []) if isinstance(data.get('image'), dict) else []

            with open(txt_filepath, 'w') as f:
                for obj in objects:
                    if 'directionindex' in obj:
                        head_x, head_y = obj['directionindex']
                        x_center_norm = max(0.0, min(1.0, head_x / img_width))
                        y_center_norm = max(0.0, min(1.0, head_y / img_height))

                        # (박스 크기)을 80픽셀로 확장 (머리+어깨)
                        head_size = 80
                        w_norm = head_size / img_width
                        h_norm = head_size / img_height

                        f.write(f"0 {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}\n")
        except Exception:
            continue

convert_to_head_hunter(os.path.join(json_base_dir, 'train'), os.path.join(label_base_dir, 'train'), "망원(훈련용)")
convert_to_head_hunter(os.path.join(json_base_dir, 'val'), os.path.join(label_base_dir, 'val'), "망원(검증용)")

yaml_content = f"train: {os.path.join(image_base_dir, 'train')}\nval: {os.path.join(image_base_dir, 'val')}\nnc: 1\nnames: ['head']" # 클래스 이름도 person에서 head로 변경!
yaml_path = os.path.join(BASE_DIR, 'datasets', 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())
print("✅ 100% 망원 헤드헌터 정답지 생성 완료! (자갈치 데이터는 과감히 버립니다)\n")

# ===================================================================
# 🚀 [Step 2] YOLO11n (Nano) 고속 훈련 시작!
# ===================================================================
!pip install -q ultralytics
import ultralytics
ultralytics.checks()
from ultralytics import YOLO

print("🔥 V6 헤드헌터 (Nano 체급) 훈련을 시작합니다...")
backup_dir = '/content/drive/MyDrive/CrowdData_Results_V6_HeadHunter'
os.makedirs(backup_dir, exist_ok=True)

# Nano 모델
model = YOLO('yolo11n.pt')

# 훈련 시작
results = model.train(
    data=yaml_path,
    epochs=40,   # [업데이트] 확실한 80% 돌파를 위해 40 에포크로 증가
    batch=32,
    imgsz=640,
    lr0=0.01,
    weight_decay=0.0005,
    patience=10,
    iou=0.5,
    mosaic=1.0,  # 모자이크 증강 최대로 설정
    mixup=0.1,   # Mixup 증강 추가
    project=backup_dir,
    name='yolo11n_v6_headhunter',
    optimizer='auto',
    val=True
)

print("확인")


Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Setup complete ✅ (12 CPUs, 83.5 GB RAM, 67.2/235.7 GB disk)
🔥 V6 헤드헌터 (Nano 체급) 훈련을 시작합니다...
Ultralytics 8.4.102 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/extracted/CrowdData/datasets/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.5, keras=False, kobj=1.0, line_width=None, l